# Strategic Decision Making with Prescriptive Analytics

In [ ]:
# Develop simulation models for business decisions

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Set seed for reproducibility
np.random.seed(42)

# 2. Defining baseline business parameters
n_simulations = 10000
base_customers = 10000
arpu = 50 # Average Revenue Per User ($ per month)

# 3. Simulate uncertain variables using probability distributions
# Churn Rate:| Normally diistributed around 15% (0,15) with a 3% stardard deviation
simulated_churn_rates = np.random.normal(loc=0.15, scale=0.03, size=n_simulations)
simulated_churn_rates = np.clip(simulated_churn_rates, 0.05, 0.35) #Realistic bounds

# New Customer Acqusitions: Modeled using a Poisson distribution ( average 1,200 new signups)
simulated_acquisitions = np.random.poisson(lam=1200, size=n_simulations)

# 4.Calculate ending customers and AAnnual Recurring Revenue (ARR) for each simulations
ending_customers = base_customers * (1 - simulated_churn_rates) + simulated_acquisitions
annual_revenue = ending_customers * arpu * 12

# 5. Statistical Risk Analysis of Simulation Results
mean_arr = np.mean(annual_revenue)
ci_lower = np.percentile(annual_revenue, 2.5) # 2.5th percentle (Downside risk)
ci_upper = np.percentile(annual_revenue, 97.5) # 97.5th percentile (Upside potential)
prob_target_miss = np.mean(annual_revenue < 6000000) # Probability ARR falls below $6.0M

print(f"--- Monte Carlo Simulation Results ({n_simulations:,} Runs) ---")
print(f"Expected Mean ARR: ${mean_arr:,.2f}")
print(f"95% Confidence Interval for ARR;  [${ci_lower:,.2f}, ${ci_upper:,.2f}]")
print(f"Probability of missing $6.0M ARR target: {prob_target_miss:.2%}")

# 6. Plotting the Risk Distribution for my Porfolio README
plt.figure(figsize=(10, 5))
plt.hist(annual_revenue / 1e6, bins=50, color='skyblue', edgecolor='black', alpha=0.8)
plt.axvline(mean_arr / 1e6, color='red', linestyle='--',linewidth=2, label=f'Mean ARR: ${mean_arr/1e6:.2f}M')
plt.axvline(ci_lower / 1e6, color='orange', linestyle=':', linewidth=2, label=f'95% Lower Bound:${ci_lower/1e6:.2f}M')
plt.title('Monte Carlo Simulation: Projected Annual Recurring Revenue 9ARR)', fontsize=13, fontweight='bold')
plt.xlabel('Annual Recurring Revenue ($ Millions)', fontsize=11)
plt.ylabel('Simulation Frequency', fontsize=11)
plt.grid(alpha=0.3, linestyle=':')
plt.tight_layout()
plt.show()

In [10]:
# Using optimization techniques for resource allocation

import numpy as np
import pandas as pd
from scipy.optimize import linprog

# 1. Simulate a customer base with churn probability, LTV, and intervention cost
np.random.seed(42)
n_customers = 50
df_opt = pd.DataFrame({
    'Customer_ID': range(1, n_customers + 1),
    'Churn_Prob': np.random.uniform(0.1, 0.9, n_customers),
    'LTV': np.random.choice([500, 1000, 2000], n_customers),
    'Intervention_Cost': 50  # Fixed cost to target a customer
})

# Campaign success rate (probability that intervention stops churn)
success_rate = 0.40

# Calculate expected net savings per customer if targeted
# Expected Value Saved = (Churn_Prob * Success_Rate * LTV) - Intervention_Cost
df_opt['Expected_Savings'] = (df_opt['Churn_Prob'] * success_rate * df_opt['LTV']) - df_opt['Intervention_Cost']

# 2. Set up the Linear Programming Optimization Problem
# We want to maximize total expected savings, which is equivalent to minimizing the negative expected savings.
# Decision variables x_i: 1 if we target customer i, 0 if we do not.
c = -df_opt['Expected_Savings'].values  # Negative because linprog minimizes

# Constraint 1: Total budget constraint (e.g., maximum budget is $1,000)
max_budget = 1000
A_ub = [df_opt['Intervention_Cost'].values]
b_ub = [max_budget]

# Bounds for each variable: binary decision (0 <= x_i <= 1)
# Note: For strict binary programming, MILP is used, but relaxed linear programming 
# provides fractional selection or sorting thresholds for large scales.
bounds = [(0, 1) for _ in range(n_customers)]

# 3. Run the Optimization
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

if result.success:
    df_opt['Allocation_Weight'] = result.x
    # Select customers where allocation weight is close to 1
    df_opt['Target_Customer'] = df_opt['Allocation_Weight'] > 0.5
    
    targeted_count = df_opt['Target_Customer'].sum()
    total_spent = targeted_count * 50
    total_projected_savings = df_opt.loc[df_opt['Target_Customer'], 'Expected_Savings'].sum()
    
    print(f"--- Optimization Results ---")
    print(f"Total Budget Available: ${max_budget}")
    print(f"Optimal Customers to Target: {targeted_count} out of {n_customers}")
    print(f"Total Budget Utilized: ${total_spent}")
    print(f"Projected Net Savings: ${total_projected_savings:,.2f}")
else:
    print("Optimization failed to find a solution.")

--- Optimization Results ---
Total Budget Available: $1000
Optimal Customers to Target: 20 out of 50
Total Budget Utilized: $1000
Projected Net Savings: $6,748.58


In [11]:
# Perform scenario analysis for risk mitigation

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Loading data to train baseline model
df = pd.read_csv(r"C:\Users\Admin\Downloads\churn-bigml-80.csv")
if 'Churn_binary' not in df.columns:
    df['Churn_binary'] = df['Churn'].map({'True': 1, 'False': 0, 'Yes': 1, 'No': 0, True: 1, False: 0})

features = ['Account length', 'Customer service calls', 'Total day minutes', 'Total eve minutes', 'International plan']
X = pd.get_dummies(df[features], drop_first=True)
y = df['Churn_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 2. Define Baseline Predictions
baseline_predictions = model.predict(X_test)
baseline_churn_count = baseline_predictions.sum()

# 3. Define Scenario: "Billing Outage / Service Spike" 
# Simulate what happens if customer service calls increase by 2 across all active accounts
X_scenario = X_test.copy()
if 'Customer service calls' in X_scenario.columns:
    X_scenario['Customer service calls'] += 2

scenario_predictions = model.predict(X_scenario)
scenario_churn_count = scenario_predictions.sum()

# 4. Quantify the Business Impact
churn_surge = scenario_churn_count - baseline_churn_count
percent_increase = (churn_surge / baseline_churn_count) * 100

print(f"--- Scenario Analysis: Customer Service Spike ---")
print(f"Baseline Predicted Churners: {baseline_churn_count}")
print(f"Scenario Predicted Churners (Calls +2): {scenario_churn_count}")
print(f"Absolute Churn Surge: +{churn_surge} customers")
print(f"Percentage Increase in Churn: +{percent_increase:.1f}%")

--- Scenario Analysis: Customer Service Spike ---
Baseline Predicted Churners: 63
Scenario Predicted Churners (Calls +2): 144
Absolute Churn Surge: +81 customers
Percentage Increase in Churn: +128.6%


In [12]:
# Integrate AI-driven recommendations into decision-making

import pandas as pd
import numpy as np

# 1. Simulate a customer dataset with churn probabilities and usage profiles
np.random.seed(42)
n_customers = 1000
df_recs = pd.DataFrame({
    'Customer_ID': range(1, n_customers + 1),
    'Churn_Prob': np.random.uniform(0.05, 0.95, n_customers),
    'Customer_Service_Calls': np.random.randint(0, 6, n_customers),
    'International_Plan': np.random.choice([0, 1], n_customers, p=[0.8, 0.2]),
    'Total_Day_Mins': np.random.uniform(100, 350, n_customers)
})

# 2. Define AI-Driven Next-Best-Action Rules Engine
def generate_ai_recommendation(row):
    prob = row['Churn_Prob']
    calls = row['Customer_Service_Calls']
    intl = row['International_Plan']
    mins = row['Total_Day_Mins']
    
    # Priority 1: High churn risk with support friction
    if prob > 0.70 and calls >= 3:
        return pd.Series([
            'High Priority', 
            'Immediate CS outreach: Schedule 1-on-1 retention call and offer 20% billing credit.'
        ])
    
    # Priority 2: High churn risk due to international or usage factors
    elif prob > 0.60 and intl == 1:
        return pd.Series([
            'Medium Priority', 
            'Pitch International Calling Bundle discount to reduce roaming cost friction.'
        ])
        
    # Priority 3: Moderate risk with heavy usage (engagement opportunity)
    elif 0.40 <= prob <= 0.60 and mins > 250:
        return pd.Series([
            'Medium Priority', 
            'Send VIP loyalty tier upgrade offer to secure long-term contract.'
        ])
        
    # Priority 4: Low risk or stable account
    else:
        return pd.Series([
            'Low Priority', 
            'Automated nurture campaign: Send monthly feature-update newsletter.'
        ])

# 3. Apply the recommendation engine across the customer base
df_recs[['Priority_Tier', 'AI_Recommendation']] = df_recs.apply(generate_ai_recommendation, axis=1)

# 4. Preview top recommended actions for customer success queue
print("--- AI-Driven Next-Best-Action Queue (Top 5 Accounts) ---")
print(df_recs[['Customer_ID', 'Churn_Prob', 'Priority_Tier', 'AI_Recommendation']].head(5).to_string(index=False))

# Summary distribution of recommendations
print("\n--- Recommendation Action Distribution ---")
print(df_recs['AI_Recommendation'].value_counts())


--- AI-Driven Next-Best-Action Queue (Top 5 Accounts) ---
 Customer_ID  Churn_Prob Priority_Tier                                                                   AI_Recommendation
           1    0.387086  Low Priority                 Automated nurture campaign: Send monthly feature-update newsletter.
           2    0.905643 High Priority Immediate CS outreach: Schedule 1-on-1 retention call and offer 20% billing credit.
           3    0.708795  Low Priority                 Automated nurture campaign: Send monthly feature-update newsletter.
           4    0.588793  Low Priority                 Automated nurture campaign: Send monthly feature-update newsletter.
           5    0.190417  Low Priority                 Automated nurture campaign: Send monthly feature-update newsletter.

--- Recommendation Action Distribution ---
AI_Recommendation
Automated nurture campaign: Send monthly feature-update newsletter.                    704
Immediate CS outreach: Schedule 1-on-1 retention ca